# 🍛 South Asian Culinary RAG Assistant
### MSc Advanced Computer Science — Text Mining Coursework
**University of Manchester**

---

This notebook runs the **full pipeline end-to-end** in order.

| Cell | Script | Purpose |
|------|--------|---------|
| 1 | — | Environment setup & dependency install |
| 2 | `web_scrape.py` | Scrape South Asian recipes → `south_asian_corpus_raw.json` |
| 3 | `enrich_metadata.py` | Tag each recipe with diet / prep_time / dish_type → `south_asian_corpus_enriched.json` |
| 4 | `recipe_creator.py` | Structure recipes into intro / ingredients / instructions → `vector_ready_corpus.json` |
| 5 | `vectorisedata.py` | Embed corpus into FAISS vector database → `faiss_index/` |
| 6 | `input_payload_creator.py` | Generate 500-query benchmark dataset → `input_payload.json` |
| 7 | `evaluate.py` | Benchmark the RAG pipeline → `output_payload_sample.json` |
| 8 | — | Display evaluation results inline |
| 9 | `app.py` | Launch Streamlit UI (live demo) |
| 10 | — | Stop Streamlit server |

**Architecture:** LangGraph · FAISS · `BAAI/bge-large-en-v1.5` · `Qwen/Qwen2.5-3B` · `Qwen/Qwen2.5-0.5B-Instruct`

> ⚠️ **Note:** Cell 3 (`enrich_metadata.py`) uses a 4-bit quantized Llama 3 8B model and is designed to run on a GPU (e.g. Kaggle). If running locally without a GPU, this step may be slow or require skipping if `south_asian_corpus_enriched.json` already exists.

---
## Cell 1 — Environment Setup

In [ ]:
import os, sys, json

HERE = os.path.dirname(os.path.abspath("__file__"))
os.chdir(HERE)
sys.path.insert(0, HERE)
print(f"Working directory: {os.getcwd()}")

# Verify all pipeline scripts are present
required_scripts = [
    "web_scrape.py",
    "enrich_metadata.py",
    "recipe_creator.py",
    "vectorisedata.py",
    "input_payload_creator.py",
    "evaluate.py",
    "assistant_core.py",
    "app.py",
]
all_ok = True
for f in required_scripts:
    exists = os.path.exists(f)
    print(f"  {'✅' if exists else '❌ MISSING'}  {f}")
    if not exists:
        all_ok = False

if not all_ok:
    raise FileNotFoundError("One or more required scripts are missing.")

print("\nInstalling / verifying packages…")
!pip install -q streamlit langchain-community langchain-huggingface \
    langgraph faiss-cpu transformers torch sentence-transformers \
    accelerate bitsandbytes tqdm beautifulsoup4 requests ollama
print("✅ All packages ready.")

---
## Cell 2 — Scrape Data

Scrapes South Asian recipe data from Wikibooks and Wikipedia using BeautifulSoup.

- Extracts introductions, ingredients, and instructions per dish
- Deduplicates content using fuzzy string matching

**Output:** `south_asian_corpus_raw.json`

In [ ]:
!python web_scrape.py

---
## Cell 3 — Enrich Metadata

Uses `NousResearch/Meta-Llama-3-8B-Instruct` (4-bit quantized) to tag each recipe with structured metadata.

- Classifies `diet` (veg / non-veg), `prep_time` (quick / slow), `dish_type` (curry / rice / bread / snack / dessert / beverage / pickle-condiment)
- Uses few-shot prompting to enforce strict JSON output
- Saves checkpoints every 25 recipes

**Input:** `south_asian_corpus_raw.json`  
**Output:** `south_asian_corpus_enriched.json`

> ⚠️ Requires GPU. Skip this cell if `south_asian_corpus_enriched.json` already exists.

In [ ]:
if os.path.exists("south_asian_corpus_enriched.json"):
    print("⏭️  south_asian_corpus_enriched.json already exists — skipping enrichment.")
else:
    !python enrich_metadata.py

---
## Cell 4 — Structure Recipes

Uses a local LLM (via Ollama) to parse raw recipe text into structured JSON.

- Extracts `intro`, `ingredients`, and `instructions` fields per dish

**Input:** `south_asian_corpus_enriched.json`  
**Output:** `vector_ready_corpus.json`

In [ ]:
!python recipe_creator.py

---
## Cell 5 — Vectorise Data

Embeds the structured corpus into a FAISS vector database using `BAAI/bge-large-en-v1.5`.

- Packages each recipe as a LangChain `Document` with metadata (diet, prep_time, dish_type)

**Input:** `vector_ready_corpus.json`  
**Output:** `faiss_index/`

> ⏱️ This may take a few minutes depending on corpus size.

In [ ]:
!python vectorisedata.py

---
## Cell 6 — Create Input Payload

Generates a 500-query benchmark dataset from the corpus.

- Includes hardcoded edge cases (non-South-Asian, vague, ingredients-only)
- Dynamically generates recipe queries from every dish using multiple templates
- Generates random metadata-based queries (diet, speed, flavor combinations)

**Input:** `vector_ready_corpus.json`  
**Output:** `input_payload.json`

In [ ]:
!python input_payload_creator.py

---
## Cell 7 — Run Benchmark Evaluation

Benchmarks the full RAG pipeline against `benchmark_dataset.json`.

Metrics computed:
- **Recall@3** — fraction of expected dishes retrieved in top-3
- **Intent Accuracy** — predicted vs expected intent
- **Latency** — wall-clock seconds per query

**Output:** `output_payload_sample.json`

> ⏱️ First run takes ~60–90s while Qwen models load. Subsequent runs are faster.

In [ ]:
!python evaluate.py

---
## Cell 8 — Display Evaluation Results

In [ ]:
import json
from IPython.display import display, Markdown

with open("output_payload_sample.json", "r", encoding="utf-8") as f:
    payload = json.load(f)

summary = payload["evaluation_summary"]
results = payload["per_query_results"]

display(Markdown("### 📊 Evaluation Summary"))
display(Markdown(
    f"| Metric | Value |\n"
    f"|--------|-------|\n"
    f"| Total Queries | {summary['total_queries']} |\n"
    f"| Mean Recall@3 | {summary.get('mean_recall@3', 'N/A')} |\n"
    f"| Intent Accuracy | {summary.get('intent_accuracy', 'N/A')} |\n"
    f"| Mean Latency (s) | {summary['mean_latency_seconds']} |\n"
    f"| Min Latency (s) | {summary['min_latency_seconds']} |\n"
    f"| Max Latency (s) | {summary['max_latency_seconds']} |\n"
))

display(Markdown("### 🔍 Per-Query Results"))
header = "| ID | Query | Expected Intent | Predicted Intent | ✓ | Recall@3 | Latency (s) |\n"
header += "|----|----|----|----|----|----|-----|\n"
rows = ""
for r in results:
    tick   = "✅" if r["intent_correct"] else "❌"
    recall = r.get("recall@3", "—")
    rows += (
        f"| {r['id']} "
        f"| {r['query'][:45]} "
        f"| {r['expected_intent']} "
        f"| {r['predicted_intent']} "
        f"| {tick} "
        f"| {recall} "
        f"| {r['latency_seconds']} |\n"
    )
display(Markdown(header + rows))

---
## Cell 9 — Launch Streamlit UI

Launches `app.py` as a background process.

After ~5 seconds, open: **http://localhost:8501**

> Run Cell 10 to stop the server when done.

In [ ]:
import subprocess, time, sys

streamlit_proc = subprocess.Popen(
    [
        sys.executable, "-m", "streamlit", "run", "app.py",
        "--server.port", "8501",
        "--server.headless", "true",
        "--server.fileWatcherType", "none",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)

time.sleep(5)
print("✅ Streamlit is running!")
print("   👉  Open your browser at:  http://localhost:8501")
print(f"   Process PID: {streamlit_proc.pid}")
print("\n   Run Cell 10 to stop the server.")

---
## Cell 10 — Stop Streamlit Server

In [ ]:
try:
    streamlit_proc.terminate()
    print(f"✅ Streamlit server (PID {streamlit_proc.pid}) stopped.")
except NameError:
    print("No active Streamlit process found in this session.")